In [ ]:
import sys
sys.path.append("../")

import torch

from npu_yolov8n import get_dataloader
from npu_yolov8n import load_model, load_NPU_model
from npu_yolov8n import demo

import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
checkpoint_path = '../checkpoints/qat_fixed.pt' 
loaded_cfg = torch.load(checkpoint_path, device, weights_only=False)['qcfg']
Nmodel = load_NPU_model(checkpoint_path, device, loaded_cfg)
Qmodel = load_model(checkpoint_path, device, model_type='qat', model_qcfg=loaded_cfg)

In [ ]:
# load dataset 
train_loader, val_loader, coco_id2label, label2coco_category = get_dataloader(data_root='../../../jongsul/yoloproject/npu_yolov8n/datasets/coco')

In [ ]:
# check NPU model output -> label.jpg, predict.jpg

# demo(train_loader, Qmodel, device, coco_id2label, label2coco_category, iou_threshold=0.01, conf_threshold=0.3, max_det=300)
demo(train_loader, Nmodel, device, coco_id2label, label2coco_category, iou_threshold=0.3, conf_threshold=0.2, max_det=300)


In [ ]:
# check for int_weight
sdict = torch.load("../outputs/int_weight.pth")
Nmodel.load_state_dict(sdict)
output = demo(train_loader, Nmodel, device, coco_id2label, label2coco_category, iou_threshold=0.3, conf_threshold=0.2, max_det=300)


In [ ]:
# get shifting bits per layer
shift_bits = {}
for k, v in Nmodel.ncfg.items():
    shift_bits[k] = v['shift']
# with open("/outputs/shifting_config.json", "w") as f:
#     json.dump(shift_bits, f, indent=2)

shift_bits    